In [23]:
import pandas as pd

In [24]:
train_df = pd.read_csv('train.csv')
drop_columns = ['Id'] + [name for name, col in train_df.items() if col.count() / len(col) < 0.6]
train_df.drop(columns=drop_columns, inplace=True)

y = train_df['SalePrice']
train_df.drop(columns=['SalePrice'], inplace=True)

In [25]:
print(train_df.shape)

(1460, 73)


## Handle int, float

In [26]:
from sklearn.impute import SimpleImputer
fill_df = train_df

int_imputer = SimpleImputer(strategy='most_frequent')
float_imputer = SimpleImputer(strategy='mean')

int_df = fill_df.select_dtypes(['int'])
fill_df[int_df.columns] = int_imputer.fit_transform(int_df)

float_df = fill_df.select_dtypes(['float'])
fill_df[float_df.columns] = float_imputer.fit_transform(float_df)

int_columns = int_df.columns.to_list()
float_columns = float_df.columns.to_list()

In [27]:
from sklearn.preprocessing import StandardScaler

num_columns = int_columns + float_columns
scaler = StandardScaler()

train_df[num_columns] = scaler.fit_transform(train_df[num_columns])

## Handle categorical features

In [28]:
print(train_df.dtypes)
cat_df = train_df.select_dtypes('object')
print(cat_df.shape)

cat_columns = cat_df.columns.to_list()

MSSubClass       float64
MSZoning          object
LotFrontage      float64
LotArea          float64
Street            object
                  ...   
MiscVal          float64
MoSold           float64
YrSold           float64
SaleType          object
SaleCondition     object
Length: 73, dtype: object
(1460, 37)


In [29]:
for name, col in cat_df.items():
    part = col.count() / len(col)
    if part != 1:
        print(name, part)

BsmtQual 0.9746575342465753
BsmtCond 0.9746575342465753
BsmtExposure 0.9739726027397261
BsmtFinType1 0.9746575342465753
BsmtFinType2 0.9739726027397261
Electrical 0.9993150684931507
GarageType 0.9445205479452055
GarageFinish 0.9445205479452055
GarageQual 0.9445205479452055
GarageCond 0.9445205479452055


In [30]:
print(cat_df.shape)

(1460, 37)


In [31]:
from sklearn.linear_model import LogisticRegression

cat_fill_models = {name: LogisticRegression(random_state=42, max_iter=200) for name in cat_columns}
for name, fill_model in cat_fill_models.items():
    notna_name_mask = train_df[name].notna()
    fill_model.fit(
        train_df.loc[notna_name_mask, num_columns],
        train_df[notna_name_mask][name]
    )


In [32]:
for name, fill_model in cat_fill_models.items():
    isna_name_mask = train_df[name].isna()

    if isna_name_mask.any():
        train_df.loc[isna_name_mask, name] = fill_model.predict(
            train_df.loc[isna_name_mask, num_columns]
        )

In [33]:
print(train_df.isna().any().any())

False


In [34]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

encoded_train_df = pd.DataFrame(
    encoder.fit_transform(train_df[cat_columns]), 
    columns=encoder.get_feature_names_out(cat_columns),
    index=train_df.index
)

train_df = pd.concat([
    train_df.drop(columns=cat_columns),
    encoded_train_df
], axis=1)

print(train_df.shape)


(1460, 266)


---

## Train

In [35]:
import numpy as np

print(train_df.shape)
X = train_df
y = np.log1p(y)

(1460, 266)


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(rmse)

0.12818140845217976


In [37]:
model = LinearRegression()
model.fit(X, y)

LinearRegression()

---

## Solution

In [38]:
test_df = pd.read_csv('test.csv')
test_df.drop(columns=drop_columns, inplace=True)

test_df[int_columns] = int_imputer.transform(test_df[int_columns])
test_df[float_columns] = float_imputer.transform(test_df[float_columns])
test_df[num_columns] = scaler.transform(test_df[num_columns])

for name, fill_model in cat_fill_models.items():
    isna_name_mask = test_df[name].isna()

    if isna_name_mask.any():
        test_df.loc[isna_name_mask, name] = fill_model.predict(
            test_df.loc[isna_name_mask, num_columns]
        )

encoded_test_df = pd.DataFrame(
    encoder.transform(test_df[cat_columns]), 
    columns=encoder.get_feature_names_out(cat_columns),
    index=test_df.index
)

test_df = pd.concat([
    test_df.drop(columns=cat_columns),
    encoded_test_df
], axis=1)

print(test_df.shape)
print(test_df.isna().any().any())

X_test = test_df

(1459, 266)
False


In [39]:
y_pred = model.predict(X_test)
y_pred = np.expm1(y_pred)

print(y_pred)

[121893.57952802 154769.81719859 184695.9924016  ... 171794.17428117
 118527.51571671 219550.46494756]


In [40]:
pd.DataFrame({
    'Id': np.arange(1461, 2919+1),
    'SalePrice': np.round(y_pred)
}).to_csv('submission.csv', index=False)